# Experimento — 4 Variáveis, 3 Classificações e Com Balanceamento

## Objetivo do Experimento

Após os testes realizados com:
- 4 classificações;
- diferentes combinações de variáveis;
- e cenários com e sem balanceamento,

foi iniciado um novo experimento utilizando:
- apenas 3 classificações;
- 4 variáveis principais;
- e aplicação de balanceamento das classes.

O principal objetivo agora é analisar como o balanceamento impacta o comportamento do modelo quando trabalhamos com uma estrutura de classes mais simplificada.


---

# O Que Será Analisado?

Durante a análise dos resultados, serão observados principalmente:

- accuracy de treino;
- accuracy de teste;
- diferença entre treino e teste;
- precision;
- recall;
- F1-score;
- comportamento da matriz de confusão;
- e impacto do balanceamento na classe `Crítica`.

Além disso, também será analisado:
- se o modelo passou a generalizar melhor;
- se houve redução do overfitting;
- e como o balanceamento alterou o comportamento das previsões.

---

# Comparações que Serão Realizadas

Os resultados deste experimento serão comparados principalmente com:

### Experimento com:
- 4 variáveis;
- 3 classificações;
- sem balanceamento.

E também com:
- o experimento anterior utilizando 4 classificações.

O objetivo dessas comparações é entender:
- qual estrutura de rótulo funciona melhor;
- qual configuração gera melhor generalização;
- e qual cenário apresenta o melhor equilíbrio entre:
  - desempenho;
  - estabilidade;
  - e capacidade de identificar padrões ambientais críticos.



## Preparação do ambiente


In [ ]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [ ]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")

    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada_3.parquet"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path("../../dataset/amostra_rotulada_3.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Adequada
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Adequada
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Adequada
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Adequada
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Adequada


In [ ]:
# preparando o ambiente para machine learning
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
# definição de x e y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Country",
        "Waterbody Type"
    ]
]

y = df["conama_status"]

In [ ]:
#train_test split

SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)


Treino: (113119, 4)
Teste: (28280, 4)


In [ ]:
# pré-processamento

categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            RandomForestClassifier(
                random_state=SEED,
                class_weight="balanced"
            )
        )
    ]
)

In [ ]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced',
                                        random_state=42))])

## Avaliação das Métricas de Treino

Antes de analisar os resultados do conjunto de teste, também foi realizada a avaliação das métricas no conjunto de treino.

Essa etapa é importante porque permite comparar o comportamento do modelo entre treino e teste, ajudando na identificação de possíveis sinais de overfitting.

Por esse motivo, não basta apenas observar uma alta acurácia no treino. O ideal é que os resultados de treino e teste permaneçam relativamente próximos, indicando que o modelo conseguiu aprender padrões mais generalizáveis ao invés de apenas memorizar os dados.

Além da acurácia, também foram analisadas métricas como precision, recall e F1-score, permitindo observar como o modelo se comporta em cada classe individualmente.

In [ ]:
y_train_pred = model.predict(X_train)

In [ ]:
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_confusion_matrix = confusion_matrix(
    y_train,
    y_train_pred
)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.8528540740282358
Train Precision:
0.89785466669279
Train Recall:
0.8528540740282358
Train F1:
0.8713236001166635
Train Confusion Matrix:
[[73076  6863  2836]
 [ 3428 22323  3487]
 [    5    26  1075]]


In [ ]:
y_pred = model.predict(X_test)

In [ ]:
# Teste com 4 variáveis - sem balanceamento
print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.691018387553041

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.83      0.80      0.81     20694
     Atenção       0.45      0.41      0.43      7309
     Crítica       0.03      0.19      0.06       277

    accuracy                           0.69     28280
   macro avg       0.44      0.47      0.43     28280
weighted avg       0.72      0.69      0.70     28280


Confusion Matrix:
[[16511  3499   684]
 [ 3420  2978   911]
 [   56   168    53]]


# Resultados Obtidos — Comparação entre Balanceamento e Quantidade de Classes

Os novos experimentos permitiram comparar diferentes configurações do problema de classificação:

* 4 classes com balanceamento;
* 3 classes sem balanceamento;
* 3 classes com balanceamento.

Esses resultados mostraram impactos importantes relacionados:

* à fragmentação das classes;
* ao desbalanceamento;
* à capacidade de generalização;
* e ao reconhecimento das classes minoritárias.

---

# Comparação entre 4 Classes e 3 Classes com Balanceamento

Ao comparar os experimentos balanceados, foi possível observar uma melhora significativa ao reduzir o problema para apenas 3 classificações.

## Resultados observados

### 4 classes com balanceamento

* treino ≈ 83%
* teste ≈ 65%

### 3 classes com balanceamento

* treino ≈ 85%
* teste ≈ 69%

A redução das classes:

* simplificou o problema;
* reduziu a fragmentação;
* e tornou as fronteiras de decisão mais estáveis.

Como consequência:

* o modelo conseguiu generalizar melhor;
* aumentou a acurácia;
* e melhorou principalmente a identificação da classe `Crítica`.

---

# Classe Crítica

A principal melhoria observada ocorreu na classe `Crítica`.

## Recall da classe crítica

### 4 classes com balanceamento

≈ 0.08

### 3 classes com balanceamento

≈ 0.19

Esse aumento foi extremamente relevante.

Isso indica que:

* o balanceamento passou a funcionar melhor;
* o modelo conseguiu prestar mais atenção nas amostras minoritárias;
* e a simplificação das classes reduziu parte da dificuldade do problema.

---

# Comparação entre 3 Classes com e sem Balanceamento

A comparação entre os experimentos com 3 classificações mostrou claramente o impacto do balanceamento no comportamento do modelo.

## Sem balanceamento

* treino ≈ 90%
* teste ≈ 74%

## Com balanceamento

* treino ≈ 85%
* teste ≈ 69%

O balanceamento reduziu parcialmente a capacidade do modelo de memorizar o conjunto dominante, mas também reduziu a acurácia geral.

Entretanto, em compensação:

* o recall da classe `Crítica` aumentou significativamente;
* o modelo passou a reconhecer mais amostras críticas;
* e houve melhora na sensibilidade às classes minoritárias.

---

# Interpretação do Balanceamento

Sem balanceamento, o modelo tende a favorecer fortemente a classe dominante (`Adequada`).

Isso faz com que:

* a acurácia geral fique alta;
* mas as classes minoritárias sejam ignoradas.

Com balanceamento:

* o modelo passa a considerar melhor as classes menores;
* aumenta a quantidade de previsões críticas;
* porém gera mais falsos positivos e reduz parte da acurácia geral.

---

# Interpretação Ambiental

Em problemas ambientais, apenas observar acurácia não é suficiente.

Em muitos casos:

* identificar corretamente uma amostra crítica pode ser mais importante do que maximizar a acurácia geral.

Por isso, o aumento do recall da classe `Crítica` se torna extremamente relevante para o projeto AquaSense.

---

# Conclusão

Os resultados indicaram que:

* reduzir para 3 classes melhorou significativamente a estabilidade do modelo;
* o balanceamento passou a funcionar melhor nesse novo cenário;
* houve melhora importante na identificação da classe crítica;
* e o problema ficou menos fragmentado.

Os próximos experimentos irão continuar avaliando:

* impacto do balanceamento;
* novas combinações de variáveis;
* e possíveis estratégias para reduzir overfitting e melhorar generalização.


In [ ]:
import pandas as pd
import plotly.graph_objects as go

# dataframe comparativo
df = pd.DataFrame({

    "Métrica": [

        "Accuracy treino",
        "Accuracy teste",
        "Overfitting",

        "Precision Excelente/Adequada",
        "Recall Excelente/Adequada",
        "F1 Excelente/Adequada",

        "Precision Boa",
        "Recall Boa",
        "F1 Boa",

        "Precision Atenção",
        "Recall Atenção",
        "F1 Atenção",

        "Precision Crítica",
        "Recall Crítica",
        "F1 Crítica"
    ],

    # ====================================
    # 3 classificações sem balanceamento
    # ====================================
    "3 Classes\nSem Balanceamento": [

        0.900,
        0.744,
        0.156,

        # Adequada
        0.81,
        0.87,
        0.84,

        # Boa não existe
        "-",
        "-",
        "-",

        # Atenção
        0.51,
        0.42,
        0.46,

        # Crítica
        0.08,
        0.03,
        0.04
    ],

    # ====================================
    # 3 classificações com balanceamento
    # ====================================
    "3 Classes\nCom Balanceamento": [

        0.853,
        0.691,
        0.162,

        # Adequada
        0.83,
        0.80,
        0.81,

        # Boa não existe
        "-",
        "-",
        "-",

        # Atenção
        0.45,
        0.41,
        0.43,

        # Crítica
        0.03,
        0.19,
        0.06
    ],

    # ====================================
    # 4 classificações com balanceamento
    # ====================================
    "4 Classes\nCom Balanceamento": [

        0.838,
        0.652,
        0.186,

        # Excelente
        0.82,
        0.79,
        0.81,

        # Boa
        0.29,
        0.23,
        0.25,

        # Atenção
        0.22,
        0.44,
        0.30,

        # Crítica
        0.05,
        0.08,
        0.07
    ]
})

# criação da tabela
fig = go.Figure(data=[go.Table(

    header=dict(
        values=list(df.columns),

        fill_color='#0F766E',

        font=dict(
            color='white',
            size=14
        ),

        align='center'
    ),

    cells=dict(
        values=[df[col] for col in df.columns],

        fill_color='#ECFEFF',

        align='center',

        font=dict(
            size=13
        ),

        height=34
    )
)])

fig.update_layout(

    title="Comparação Experimental — Balanceamento e Quantidade de Classes",

    width=1350,

    height=750
)

fig.show()